In [4]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Column descriptions provide semantic context for embedding-based mapping.
# Set USE_COLUMN_DESCRIPTIONS = False to run in baseline mode (name + samples only).
# COLUMN_DESCRIPTIONS = {
#     "column_1": "Description_1",
#     "column_2": "Description_2",
#     "column_3": "Description_3",
# }
COLUMN_DESCRIPTIONS = {
    "n° US": "Unique identifier of the Stratigraphic Unit (SU).",
    "Localisation": "Spatial location of the SU within the excavation site.",
    "Description": "Textual description of the nature and physical characteristics of the SU.",
    "Etendue": "Spatial extent or area covered by the SU.",
    "Antérieur à": "List of SUs that are stratigraphically posterior to this one.",
    "Postérieur à": "List of SUs that are stratigraphically anterior to this one.",
    "Egal à": "SUs considered equivalent or contemporaneous.",
    "Etat": "Conservation state or classification.",
    "Mortier": "Nature and characteristics of the mortar.",
}

USE_COLUMN_DESCRIPTIONS = True

# Upper bounds for the automatic top-k grid search.
# Embedding inference is performed once at (MAX_TOP_K_CLASSES, MAX_TOP_K_PROPERTIES);
# validation is then replayed cheaply for every (k_c, k_p) in the search space.
MAX_TOP_K_CLASSES    = 15   # upper bound for k_classes
MAX_TOP_K_PROPERTIES = 30   # upper bound for k_properties

# Parsimony threshold ε: accept (k_c, k_p) if its coverage ≥ max_coverage × (1 − ε).
# 0.0 enforces the strict maximum; raise to e.g. 0.02 to prefer smaller k values.
PARSIMONY_THRESHOLD  = 0.0

# INPUT_CSV_FILES = ["../data/input/data.csv"]
INPUT_CSV_FILES = ["../data/input/us.csv"]


In [5]:
import sys
import os
import csv
import pandas as pd
import numpy as np
import json
from collections import deque
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Set
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline


In [6]:
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

CIDOC_CLASSES_FILE    = "../data/cidoc_classes.txt"
CIDOC_PROPERTIES_FILE = "../data/cidoc_properties.txt"
CIDOC_OWL_FILE        = "../data/CIDOC_CRM_v7.1.3.owl"
CIDOC_RDF_FILE        = "../data/CIDOC_CRM_v7.1.3.rdf"
INPUT_DIR             = "../data/input"


In [7]:
for csv_path in INPUT_CSV_FILES:
    status = "OK" if os.path.exists(csv_path) else "MISSING"
    print(f"[{status}] {os.path.basename(csv_path)}")


[OK] us.csv


In [8]:
def load_cidoc_classes(file_path: str) -> List[Dict]:
    """
    Parse the CIDOC-CRM class definitions from a plain-text file.

    Each line is expected to follow the format:
        # E<n> <English label> / <French label> – <scope note>

    Parameters
    ----------
    file_path : str
        Path to the CIDOC classes text file.

    Returns
    -------
    List[Dict]
        Each entry contains 'code', 'label', 'description', and 'full_name'.
    """
    classes = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not (line and line.startswith('# E')):
                continue
            parts = line[2:].split(' ', 1)
            if len(parts) < 2:
                continue
            code_part = parts[0].strip()
            rest      = parts[1].strip()
            if ' – ' not in rest:
                continue
            label_part, description = rest.split(' – ', 1)
            label = label_part.split('/')[0].strip()
            code  = f"{code_part}_{label.replace(' ', '_')}"
            classes.append({
                'code':      code,
                'label':     label,
                'description': description.strip(),
                'full_name': f"{code} {label}",
            })
    return classes


def load_cidoc_properties(file_path: str) -> List[Dict]:
    """
    Parse the CIDOC-CRM property definitions from a plain-text file.

    Each line is expected to follow the format:
        # P<n> <English label> / <French label> – <description> [Domain: …] [Range: …]

    Parameters
    ----------
    file_path : str
        Path to the CIDOC properties text file.

    Returns
    -------
    List[Dict]
        Each entry contains 'code', 'label', 'description', 'domain', 'range',
        and 'full_name'.
    """
    properties = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not (line and line.startswith('# P')):
                continue
            parts = line[2:].split(' – ', 2)
            if len(parts) < 3:
                continue
            code             = parts[0].strip()
            label            = parts[1].split(' / ')[0].strip()
            description_part = parts[2]

            domain = range_val = None
            if '[Domain:' in description_part:
                s = description_part.find('[Domain:') + 8
                domain = description_part[s:description_part.find(']', s)].strip()
            if '[Range:' in description_part:
                s = description_part.find('[Range:') + 7
                range_val = description_part[s:description_part.find(']', s)].strip()

            desc_end    = description_part.find('[Domain:')
            description = description_part[:desc_end].strip() if desc_end > 0 else description_part.strip()

            properties.append({
                'code':        code,
                'label':       label,
                'description': description,
                'domain':      domain,
                'range':       range_val,
                'full_name':   f"{code} {label}",
            })
    return properties


def detect_encoding(file_path: str) -> str:
    """
    Detect the character encoding of a text file by attempting a ranked list
    of common encodings. Falls back to 'latin-1' if none succeed.

    Parameters
    ----------
    file_path : str

    Returns
    -------
    str
        Detected encoding name.
    """
    for enc in ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'windows-1252']:
        try:
            with open(file_path, 'r', encoding=enc) as f:
                f.read()
            return enc
        except (UnicodeDecodeError, UnicodeError):
            continue
    return 'latin-1'


def detect_delimiter(file_path: str, encoding: str = 'latin-1') -> str:
    """
    Infer the delimiter of a CSV file using :class:`csv.Sniffer`.

    Parameters
    ----------
    file_path : str
    encoding  : str

    Returns
    -------
    str
        Detected delimiter character; defaults to ',' on failure.
    """
    with open(file_path, 'r', encoding=encoding) as f:
        first_line = f.readline()
    sniffer = csv.Sniffer()
    try:
        return sniffer.sniff(first_line).delimiter
    except csv.Error:
        return ','


In [9]:
cidoc_classes     = load_cidoc_classes(CIDOC_CLASSES_FILE)
cidoc_properties  = load_cidoc_properties(CIDOC_PROPERTIES_FILE)
print(f"Loaded {len(cidoc_classes)} CIDOC classes and {len(cidoc_properties)} CIDOC properties.")


Loaded 75 CIDOC classes and 309 CIDOC properties.


In [10]:
from sentence_transformers import SentenceTransformer


class MultilingualEmbedder:
    """
    Thin wrapper around a :class:`SentenceTransformer` model that provides
    consistent encode interfaces used throughout the pipeline.

    The default model (paraphrase-multilingual-mpnet-base-v2) supports both
    French and English inputs, which is required for multilingual column
    headers and CIDOC-CRM scope notes.

    Parameters
    ----------
    model_name : str
        Hugging Face model identifier.
    """

    def __init__(self, model_name: str = "paraphrase-multilingual-mpnet-base-v2"):
        self.model      = SentenceTransformer(model_name)
        self.model_name = model_name
        print(f"Embedder loaded: {model_name}")

    def encode_text(self, text: str) -> np.ndarray:
        """Encode a single string into a dense vector."""
        return self.model.encode(text, convert_to_numpy=True)

    def encode_texts(self, texts: List[str]) -> np.ndarray:
        """Encode a list of strings into a 2-D embedding matrix."""
        return self.model.encode(texts, convert_to_numpy=True, show_progress_bar=True)


try:
    embedder            = MultilingualEmbedder()
    EMBEDDINGS_AVAILABLE = True
except Exception as e:
    print(f"Failed to load embedder: {e}")
    EMBEDDINGS_AVAILABLE = False
    embedder             = None


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedder loaded: paraphrase-multilingual-mpnet-base-v2


## Phase 1 and 2: Class and Property Selection

In [11]:
def phase1_class_selection(
    df: pd.DataFrame,
    cidoc_classes: List[Dict],
    embedder: MultilingualEmbedder,
    top_k: int = 5,
    model_name: str = "paraphrase-multilingual-mpnet-base-v2",
    use_descriptions: bool = False,
    column_descriptions: Optional[Dict[str, str]] = None,
) -> tuple:
    """
    Phase 1 — CIDOC-CRM class candidate retrieval via semantic embedding.

    For each column in *df*, a textual representation is constructed from the
    column header, optional semantic description, and sample values.  Cosine
    similarity is computed against all CIDOC-CRM class embeddings, and the
    top-*k* candidates are retained.  The rank-1 candidate is designated the
    selected class.

    Parameters
    ----------
    df : pd.DataFrame
        Source dataset whose columns are to be mapped.
    cidoc_classes : List[Dict]
        CIDOC-CRM class registry loaded by :func:`load_cidoc_classes`.
    embedder : MultilingualEmbedder
        Sentence embedding model.
    top_k : int
        Number of candidate classes to retain per column.
    model_name : str
        Model identifier (stored in results for traceability).
    use_descriptions : bool
        If True, inject the corresponding entry from *column_descriptions*
        into the column's textual representation.
    column_descriptions : Optional[Dict[str, str]]
        Mapping from column name to a domain-expert description.

    Returns
    -------
    column_results : Dict[str, Dict]
        Per-column mapping results including selected class, confidence score,
        embedding text, and full top-*k* candidate list.
    avg_confidence : float
        Mean cosine similarity of the selected (rank-1) classes across all columns.
    """
    class_texts      = [f"{cls['full_name']}: {cls['description']}" for cls in cidoc_classes]
    class_embeddings = np.array(embedder.encode_texts(class_texts))

    column_results = {}

    for column in df.columns:
        sample_values = df[column].dropna().astype(str).head(3).tolist()

        if use_descriptions and column_descriptions and column in column_descriptions:
            column_text = (
                f"Column name: {column}. "
                f"Description: {column_descriptions[column]} "
                f"Sample values: {', '.join(sample_values)}."
            )
        else:
            column_text = (
                f"Column name: {column}. "
                f"Sample values: {', '.join(sample_values)}."
            )

        col_emb      = np.array(embedder.encode_text(column_text))
        similarities = np.dot(class_embeddings, col_emb) / (
            np.linalg.norm(class_embeddings, axis=1) * np.linalg.norm(col_emb)
        )

        top_indices  = np.argsort(similarities)[::-1][:top_k]
        top_classes  = [cidoc_classes[i] for i in top_indices]
        top_scores   = [float(similarities[i]) for i in top_indices]

        best_class, best_score = top_classes[0], top_scores[0]

        column_results[column] = {
            'selected_class':  best_class['code'],
            'selected_label':  best_class['label'],
            'confidence':      best_score,
            'justification':   f"Embedding-based selection (score: {best_score:.4f})",
            'embedding_text':  column_text,
            'top_candidates':  [
                {'code': cls['code'], 'label': cls['label'], 'score': score}
                for cls, score in zip(top_classes, top_scores)
            ],
            'sample_values':   sample_values,
        }

    avg_confidence = float(np.mean([r['confidence'] for r in column_results.values()]))
    print(f"Phase 1 complete — {len(column_results)} columns mapped, "
          f"mean confidence: {avg_confidence:.4f}")
    return column_results, avg_confidence


In [12]:
def phase2_property_selection(
    phase1_results: Dict[str, Dict],
    cidoc_properties: List[Dict],
    embedder: MultilingualEmbedder,
    top_k: int = 5,
    use_descriptions: bool = False,
    column_descriptions: Optional[Dict[str, str]] = None,
) -> Dict[str, Dict]:
    """
    Phase 2 — CIDOC-CRM property candidate retrieval via semantic embedding.

    For every column mapped in Phase 1, a relational text is constructed that
    describes the expected property linking the fixed source class
    (E22_Human-Made_Object, representing the stratigraphic unit) to the target
    column.  Cosine similarity is computed against all CIDOC-CRM property
    embeddings and the top-*k* candidates are retained.

    Parameters
    ----------
    phase1_results : Dict[str, Dict]
        Output of :func:`phase1_class_selection`.
    cidoc_properties : List[Dict]
        CIDOC-CRM property registry loaded by :func:`load_cidoc_properties`.
    embedder : MultilingualEmbedder
        Sentence embedding model.
    top_k : int
        Number of candidate properties to retain per relation.
    use_descriptions : bool
        If True, inject domain-expert descriptions into the relational text.
    column_descriptions : Optional[Dict[str, str]]
        Mapping from column name to a domain-expert description.

    Returns
    -------
    Dict[str, Dict]
        Per-relation mapping results keyed as 'E22_to_<column>'.  Each entry
        contains the selected property, confidence score, relational text,
        expected domain/range, and the full top-*k* candidate list.
    """
    SOURCE_CLASS = "E22_Human-Made_Object"

    prop_texts        = [f"{p['full_name']}: {p['description']}" for p in cidoc_properties]
    prop_embeddings   = np.array(embedder.encode_texts(prop_texts))

    property_results = {}

    for target_column in phase1_results:
        target_samples = phase1_results[target_column].get('sample_values', [])[:2]

        if use_descriptions and column_descriptions and target_column in column_descriptions:
            relation_text = (
                f"Property linking a stratigraphic unit (E22_Human-Made_Object) "
                f"to {target_column}. "
                f"Description of {target_column}: {column_descriptions[target_column]} "
                f"Sample values of {target_column}: {', '.join(target_samples)}."
            )
        else:
            relation_text = (
                f"Property linking a stratigraphic unit (E22_Human-Made_Object) "
                f"to {target_column}. "
                f"Sample values of {target_column}: {', '.join(target_samples)}."
            )

        rel_emb       = np.array(embedder.encode_text(relation_text))
        similarities  = np.dot(prop_embeddings, rel_emb) / (
            np.linalg.norm(prop_embeddings, axis=1) * np.linalg.norm(rel_emb)
        )

        top_indices   = np.argsort(similarities)[::-1][:top_k]
        top_props     = [cidoc_properties[i] for i in top_indices]
        top_scores    = [float(similarities[i]) for i in top_indices]

        best_prop, best_score = top_props[0], top_scores[0]

        property_results[f"E22_to_{target_column}"] = {
            'source_column':   f"{SOURCE_CLASS} (Stratigraphic Unit)",
            'target_column':   target_column,
            'source_class':    SOURCE_CLASS,
            'selected_property': best_prop['code'],
            'selected_label':  best_prop['full_name'],
            'confidence':      best_score,
            'justification':   f"Embedding-based selection (score: {best_score:.4f})",
            'relation_text':   relation_text,
            'domain_expected': best_prop['domain'],
            'range_expected':  best_prop['range'],
            'top_candidates':  [
                {
                    'code':        p['code'],
                    'label':       p['full_name'],
                    'score':       score,
                    'domain':      p['domain'],
                    'range':       p['range'],
                    'description': p['description'],
                }
                for p, score in zip(top_props, top_scores)
            ],
            'target_samples':  target_samples,
        }

    avg_conf = float(np.mean([r['confidence'] for r in property_results.values()])) if property_results else 0.0
    print(f"Phase 2 complete — {len(property_results)} relations mapped, "
          f"mean confidence: {avg_conf:.4f}")
    return property_results


In [13]:
class ClassHierarchyUtil:
    """
    Navigation and querying utilities for the CIDOC-CRM class hierarchy.

    The hierarchy is represented as a dictionary mapping each class code to
    the list of its *direct* superclasses.  Transitive closure is computed
    on demand via depth-first traversal and cached to avoid repeated work.

    Parameters
    ----------
    class_hierarchy : Dict[str, List[str]]
        Adjacency list: ``{class_code: [direct_superclass_code, …]}``.
    """

    def __init__(self, class_hierarchy: Dict[str, List[str]]):
        self.class_hierarchy  = class_hierarchy
        self._subclass_cache: Dict[str, List[str]] = {}

        # Build short-code aliases (e.g. 'E22' → 'E22_Human-Made_Object').
        self.class_aliases: Dict[str, str] = {}
        for full_code in class_hierarchy:
            self.class_aliases[full_code] = full_code
            short = full_code.split('_', 1)[0]
            if short not in self.class_aliases:
                self.class_aliases[short] = full_code

        print(f"ClassHierarchyUtil initialised : {len(class_hierarchy)} classes.")

    def normalize_class_code(self, class_code: str) -> str:
        """Resolve a short code (e.g. 'E22') to its canonical full code."""
        return self.class_aliases.get(class_code, class_code) if class_code else class_code

    def get_all_superclasses(self, class_code: str, visited: Optional[Set[str]] = None) -> List[str]:
        """
        Return all transitive superclasses of *class_code* via DFS.

        Parameters
        ----------
        class_code : str
        visited    : Optional[Set[str]]
            Tracks visited nodes to prevent infinite recursion on cycles.

        Returns
        -------
        List[str]
            Deduplicated list of all ancestor class codes.
        """
        class_code = self.normalize_class_code(class_code)
        if visited is None:
            visited = set()
        if class_code in visited:
            return []
        visited.add(class_code)

        result = []
        for parent in self.class_hierarchy.get(class_code, []):
            result.append(parent)
            result.extend(self.get_all_superclasses(parent, visited))
        return list(set(result))

    def get_all_subclasses(self, class_code: str) -> List[str]:
        """
        Return all transitive subclasses of *class_code*.

        Results are cached after the first computation.

        Parameters
        ----------
        class_code : str

        Returns
        -------
        List[str]
        """
        class_code = self.normalize_class_code(class_code)
        if class_code in self._subclass_cache:
            return self._subclass_cache[class_code]
        result = [
            cls for cls in self.class_hierarchy
            if class_code in self.get_all_superclasses(cls)
        ]
        self._subclass_cache[class_code] = result
        return result

    def is_same_or_subclass(self, class_a: str, class_b: str) -> bool:
        """
        Return True if *class_a* is identical to or a subclass of *class_b*.

        Parameters
        ----------
        class_a : str
        class_b : str

        Returns
        -------
        bool
        """
        class_a = self.normalize_class_code(class_a)
        class_b = self.normalize_class_code(class_b)
        return class_a == class_b or class_b in self.get_all_superclasses(class_a)

    def get_hierarchy_path(self, class_a: str, class_b: str) -> Optional[List[str]]:
        """
        Return the shortest ancestry path from *class_a* up to *class_b* using BFS.

        Parameters
        ----------
        class_a : str  Starting class (more specific).
        class_b : str  Target ancestor class.

        Returns
        -------
        Optional[List[str]]
            Ordered list [class_a, …, class_b], or None if no path exists.
        """
        class_a = self.normalize_class_code(class_a)
        class_b = self.normalize_class_code(class_b)
        if class_a == class_b:
            return [class_a]

        queue   = deque([(class_a, [class_a])])
        visited: Set[str] = set()

        while queue:
            current, path = queue.popleft()
            if current in visited:
                continue
            visited.add(current)
            for parent in self.class_hierarchy.get(current, []):
                new_path = path + [parent]
                if parent == class_b:
                    return new_path
                queue.append((parent, new_path))
        return None


In [14]:
class DomainValidator:
    """
    Validates whether a candidate subject class satisfies the domain constraint
    of a CIDOC-CRM property.

    A candidate is valid if it is identical to the required domain class or is
    one of its subclasses in the CIDOC-CRM hierarchy.

    Parameters
    ----------
    hierarchy_util : ClassHierarchyUtil
    """

    def __init__(self, hierarchy_util: ClassHierarchyUtil):
        self.hierarchy_util = hierarchy_util

    def validate_domain(self, candidate_class: str, property_domain: str) -> Dict:
        """
        Check whether *candidate_class* satisfies the domain constraint.

        Parameters
        ----------
        candidate_class : str
            Class proposed as the subject (e.g. 'E22').
        property_domain : str
            Required domain class of the property (e.g. 'E71').

        Returns
        -------
        Dict
            Keys: ``valid`` (bool), ``match_type`` ('exact' | 'subclass' | 'invalid'),
            ``explanation`` (str), ``hierarchy_path`` (List[str] | None).
        """
        if not property_domain:
            return {
                'valid': False, 'match_type': 'invalid',
                'explanation': 'No domain specified for this property.',
                'hierarchy_path': None,
            }
        if candidate_class == property_domain:
            return {
                'valid': True, 'match_type': 'exact',
                'explanation': f"{candidate_class} exactly matches required domain {property_domain}.",
                'hierarchy_path': [candidate_class],
            }
        if self.hierarchy_util.is_same_or_subclass(candidate_class, property_domain):
            path = self.hierarchy_util.get_hierarchy_path(candidate_class, property_domain)
            return {
                'valid': True, 'match_type': 'subclass',
                'explanation': f"{candidate_class} is a subclass of {property_domain}.",
                'hierarchy_path': path,
            }
        return {
            'valid': False, 'match_type': 'invalid',
            'explanation': f"{candidate_class} is neither equal to nor a subclass of {property_domain}.",
            'hierarchy_path': None,
        }


In [15]:
class RangeValidator:
    """
    Validates whether a candidate object class satisfies the range constraint
    of a CIDOC-CRM property.

    Validation logic is symmetric to :class:`DomainValidator`.

    Parameters
    ----------
    hierarchy_util : ClassHierarchyUtil
    """

    def __init__(self, hierarchy_util: ClassHierarchyUtil):
        self.hierarchy_util = hierarchy_util

    def validate_range(self, object_class: str, property_range: str) -> Dict:
        """
        Check whether *object_class* satisfies the range constraint.

        Parameters
        ----------
        object_class    : str
            Class proposed as the object (e.g. 'E53').
        property_range  : str
            Required range class of the property (e.g. 'E41').

        Returns
        -------
        Dict
            Keys: ``valid``, ``match_type``, ``explanation``, ``hierarchy_path``.
        """
        if not property_range:
            return {
                'valid': False, 'match_type': 'invalid',
                'explanation': 'No range specified for this property.',
                'hierarchy_path': None,
            }
        if object_class == property_range:
            return {
                'valid': True, 'match_type': 'exact',
                'explanation': f"{object_class} exactly matches required range {property_range}.",
                'hierarchy_path': [object_class],
            }
        if self.hierarchy_util.is_same_or_subclass(object_class, property_range):
            path = self.hierarchy_util.get_hierarchy_path(object_class, property_range)
            return {
                'valid': True, 'match_type': 'subclass',
                'explanation': f"{object_class} is a subclass of {property_range}.",
                'hierarchy_path': path,
            }
        return {
            'valid': False, 'match_type': 'invalid',
            'explanation': f"{object_class} is neither equal to nor a subclass of {property_range}.",
            'hierarchy_path': None,
        }


In [16]:
class PropertyValidator:
    """
    Combined domain and range validator for CIDOC-CRM properties.

    Given a subject class, property code, and object class, this validator
    determines whether the triple (subject, property, object) is
    ontologically consistent with the CIDOC-CRM specification.

    Parameters
    ----------
    hierarchy_util : ClassHierarchyUtil
    properties     : Dict[str, Dict]
        Property registry mapping property codes to their metadata
        (domain, range, label, description).
    """

    def __init__(self, hierarchy_util: ClassHierarchyUtil, properties: Dict[str, Dict]):
        self.hierarchy_util   = hierarchy_util
        self.properties       = properties
        self.domain_validator = DomainValidator(hierarchy_util)
        self.range_validator  = RangeValidator(hierarchy_util)
        print(f"PropertyValidator ready {len(properties)} properties registered.")

    def validate_property(self, subject_class: str, property_code: str, object_class: str) -> Dict:
        """
        Validate whether *property_code* can be used between *subject_class*
        and *object_class* according to the CIDOC-CRM ontology.

        Parameters
        ----------
        subject_class  : str  Candidate subject (domain) class.
        property_code  : str  CIDOC-CRM property code (e.g. 'P45').
        object_class   : str  Candidate object (range) class.

        Returns
        -------
        Dict
            Keys: ``valid`` (bool), ``domain_ok`` (bool), ``range_ok`` (bool),
            ``messages`` (List[str]), ``domain_result`` (Dict),
            ``range_result`` (Dict), ``property_info`` (Dict).
        """
        if property_code not in self.properties:
            return {
                'valid': False, 'domain_ok': False, 'range_ok': False,
                'messages': [f"Property {property_code} not found in the ontology."],
                'domain_result': None, 'range_result': None,
            }
        prop_info = self.properties[property_code]
        messages  = []

        domain_result = self.domain_validator.validate_domain(subject_class, prop_info.get('domain'))
        range_result  = self.range_validator.validate_range(object_class,  prop_info.get('range'))

        messages.append(
            f"[DOMAIN {'OK' if domain_result['valid'] else 'INVALID'}] "
            f"{domain_result['explanation']}"
        )
        messages.append(
            f"[RANGE {'OK' if range_result['valid'] else 'INVALID'}] "
            f"{range_result['explanation']}"
        )

        return {
            'valid':         domain_result['valid'] and range_result['valid'],
            'domain_ok':     domain_result['valid'],
            'range_ok':      range_result['valid'],
            'messages':      messages,
            'domain_result': domain_result,
            'range_result':  range_result,
            'property_info': prop_info,
        }

    def find_valid_properties(
        self,
        subject_class: str,
        object_class: str,
        max_results: int = 10,
    ) -> List[Dict]:
        """
        Enumerate all ontologically valid CIDOC-CRM properties between two classes.

        Results are ranked by a score that rewards exact over subclass matches
        (2 points for exact, 1 for subclass) on both domain and range.

        Parameters
        ----------
        subject_class : str
        object_class  : str
        max_results   : int

        Returns
        -------
        List[Dict]
            Sorted list of valid property entries, each containing
            'property_code', 'property_label', 'score', 'domain_match',
            and 'range_match'.
        """
        results = []
        for prop_code, prop_info in self.properties.items():
            r = self.validate_property(subject_class, prop_code, object_class)
            if not r['valid']:
                continue
            score = (
                (2 if r['domain_result']['match_type'] == 'exact' else 1) +
                (2 if r['range_result']['match_type']  == 'exact' else 1)
            )
            results.append({
                'property_code':  prop_code,
                'property_label': prop_info.get('label', ''),
                'score':          score,
                'domain_match':   r['domain_result']['match_type'],
                'range_match':    r['range_result']['match_type'],
            })
        results.sort(key=lambda x: x['score'], reverse=True)
        return results[:max_results]


In [17]:
class CIDOCOntologyLoader:
    """
    Lightweight loader for the CIDOC-CRM ontology.

    Constructs a class hierarchy dictionary from the RDF/OWL file using
    :mod:`xml.etree.ElementTree`.  Falls back to an empty hierarchy
    (exact-match validation only) if the RDF file is unavailable or
    cannot be parsed.

    Parameters
    ----------
    rdf_file : str, optional
        Path to the CIDOC-CRM RDF/OWL file.
    """

    def __init__(self, rdf_file: Optional[str] = None):
        self.rdf_file = rdf_file

    @staticmethod
    def _extract_class_token(uri: str) -> str:
        """Extract the local name from a URI (e.g. '…#E22_Human-Made_Object' → 'E22_Human-Made_Object')."""
        if not uri:
            return ""
        value = str(uri).strip()
        if '#' in value:
            return value.split('#')[-1]
        if '/' in value:
            return value.rstrip('/').split('/')[-1]
        return value

    @staticmethod
    def _build_aliases(class_codes: List[str]) -> Dict[str, str]:
        """Map short codes (e.g. 'E22') to their full canonical codes."""
        aliases: Dict[str, str] = {}
        for full in class_codes:
            aliases[full] = full
            short = full.split('_', 1)[0]
            if short not in aliases:
                aliases[short] = full
        return aliases

    def build_complete_ontology(self) -> tuple:
        """
        Construct the class hierarchy, property registry, and class index
        from the pre-loaded CIDOC-CRM data.

        The hierarchy is extracted from ``rdfs:subClassOf`` triples in the
        RDF file.  The property registry and class index reuse the data
        already parsed by :func:`load_cidoc_classes` and
        :func:`load_cidoc_properties` (module-level variables).

        Returns
        -------
        class_hierarchy : Dict[str, List[str]]
            ``{class_code: [direct_superclass_code, …]}``.
        properties      : Dict[str, Dict]
            ``{property_code: {code, label, description, domain, range, full_name}}``.
        classes         : Dict[str, Dict]
            ``{class_code: class_dict}``.
        """
        class_hierarchy = {cls['code']: [] for cls in cidoc_classes}
        aliases         = self._build_aliases(list(class_hierarchy.keys()))
        links_added     = 0

        if self.rdf_file and os.path.exists(self.rdf_file):
            try:
                ns = {
                    'rdf':  'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
                    'rdfs': 'http://www.w3.org/2000/01/rdf-schema#',
                    'owl':  'http://www.w3.org/2002/07/owl#',
                }
                rdf_about    = f"{{{ns['rdf']}}}about"
                rdf_resource = f"{{{ns['rdf']}}}resource"

                root = ET.parse(self.rdf_file).getroot()
                class_nodes  = (
                    root.findall('.//owl:Class', ns) +
                    root.findall('.//rdfs:Class', ns)
                )

                for node in class_nodes:
                    child_token = self._extract_class_token(node.attrib.get(rdf_about, ''))
                    child_code  = aliases.get(child_token, child_token)
                    if not child_code:
                        continue
                    class_hierarchy.setdefault(child_code, [])

                    for sub in node.findall('rdfs:subClassOf', ns):
                        parent_token = self._extract_class_token(sub.attrib.get(rdf_resource, ''))
                        parent_code  = aliases.get(parent_token, parent_token)
                        if not parent_code or parent_code == child_code:
                            continue
                        class_hierarchy.setdefault(parent_code, [])
                        if parent_code not in class_hierarchy[child_code]:
                            class_hierarchy[child_code].append(parent_code)
                            links_added += 1

            except Exception as exc:
                print(f"Warning: RDF parsing failed — falling back to exact-match validation. ({exc})")

        properties = {
            p['code']: {
                'code':        p['code'],
                'label':       p['label'],
                'description': p.get('description', ''),
                'domain':      p.get('domain', ''),
                'range':       p.get('range', ''),
                'full_name':   p.get('full_name', f"{p['code']} {p['label']}"),
            }
            for p in cidoc_properties
        }
        classes = {cls['code']: cls for cls in cidoc_classes}

        print(f"Ontology built : {len(class_hierarchy)} classes, "
              f"{len(properties)} properties, {links_added} subClassOf links.")
        return class_hierarchy, properties, classes


In [18]:
if not os.path.exists(CIDOC_RDF_FILE):
    raise FileNotFoundError(f"CIDOC-CRM RDF file not found: {CIDOC_RDF_FILE}")

ontology_loader = CIDOCOntologyLoader(CIDOC_RDF_FILE)
class_hierarchy_cidoc, properties_cidoc, classes_cidoc = ontology_loader.build_complete_ontology()

hierarchy_util     = ClassHierarchyUtil(class_hierarchy_cidoc)
domain_validator   = DomainValidator(hierarchy_util)
range_validator    = RangeValidator(hierarchy_util)
property_validator = PropertyValidator(hierarchy_util, properties_cidoc)

print("Validation system initialised successfully.")


Ontology built : 76 classes, 309 properties, 89 subClassOf links.
ClassHierarchyUtil initialised : 76 classes.
PropertyValidator ready 309 properties registered.
Validation system initialised successfully.


In [19]:
dataframes: Dict[str, pd.DataFrame] = {}

for csv_path in INPUT_CSV_FILES:
    if not os.path.exists(csv_path):
        print(f"[MISSING] {csv_path}")
        continue
    name = os.path.basename(csv_path)
    try:
        df = pd.read_csv(csv_path)
        dataframes[name] = df
        print(f"[OK] {name} : {df.shape[0]} rows × {df.shape[1]} columns")
    except Exception as exc:
        print(f"[ERROR] {name}: {exc}")


[OK] us.csv : 181 rows × 11 columns


In [20]:
def validate_multi_file_mappings(
    all_results:       Dict,
    domain_validator:  DomainValidator,
    range_validator:   RangeValidator,
    hierarchy_util:    ClassHierarchyUtil,
    ontology_loader:   CIDOCOntologyLoader,
    properties_cidoc:  Optional[Dict] = None,
) -> Dict:
    """
    Validate CIDOC-CRM mappings produced by :func:`process_all_csv_files`.

    For each column (relation), every property in the top-*k* candidate list
    is checked for domain validity against the fixed source class
    (E22_Human-Made_Object).  Range validity is checked against *all* top-*k*
    class candidates returned by Phase 1 for the target column.  A relation
    is considered valid if at least one (property, target-class) pair satisfies
    both constraints simultaneously.

    Parameters
    ----------
    all_results      : Dict  Output of :func:`process_all_csv_files`.
    domain_validator : DomainValidator
    range_validator  : RangeValidator
    hierarchy_util   : ClassHierarchyUtil
    ontology_loader  : CIDOCOntologyLoader
    properties_cidoc : Optional[Dict]
        Property registry; resolved from globals if not provided.

    Returns
    -------
    Dict
        Per-file validation results containing 'validation_data' (list of
        per-candidate records) and 'stats' (aggregate validation statistics).
    """
    if properties_cidoc is None:
        properties_cidoc = globals().get('properties_cidoc', {})

    validation_results: Dict = {}

    for filename, results in all_results.items():
        if 'error' in results or 'phase2_results' not in results:
            print(f"Skipping {filename} (no results to validate).")
            continue

        phase1_results = results.get('phase1_results', {})
        phase2_results = results.get('phase2_results', {})

        validation_data           = []
        relation_validity: Dict[str, bool] = {}
        selected_candidate_validity: List[bool] = []
        total_range_checks = valid_range_checks = 0

        for relation_key, prop_info in phase2_results.items():
            if '_to_' not in relation_key or not isinstance(prop_info, dict):
                continue

            parts             = relation_key.split('_to_', 1)
            source_class_code = prop_info.get('source_class', parts[0])
            target_column     = parts[1]

            target_class_info      = phase1_results.get(target_column, {})
            selected_target_code   = target_class_info.get('selected_class', 'Unknown')
            selected_target_label  = target_class_info.get('selected_label', selected_target_code)

            target_top_classes = target_class_info.get('top_candidates', [{
                'code':  selected_target_code,
                'label': selected_target_label,
                'score': target_class_info.get('confidence', 0.0),
            }])

            top_candidates = prop_info.get('top_candidates', [{
                'code':   prop_info.get('selected_property', 'Unknown'),
                'label':  prop_info.get('selected_label', ''),
                'score':  prop_info.get('confidence', 0.0),
                'domain': prop_info.get('domain_expected', 'Unknown'),
                'range':  prop_info.get('range_expected', 'Unknown'),
            }])

            selected_property_code        = prop_info.get('selected_property', 'Unknown')
            relation_validity[relation_key] = False

            for cand_rank, candidate in enumerate(top_candidates, 1):
                if not isinstance(candidate, dict):
                    continue

                prop_code  = candidate.get('code') or selected_property_code or 'Unknown'
                prop_label = (candidate.get('label') or candidate.get('full_name')
                              or prop_info.get('selected_label', ''))
                prop_conf  = float(candidate.get('score', prop_info.get('confidence', 0.0)))

                ont_info   = properties_cidoc.get(prop_code, {})
                prop_domain = (ont_info.get('domain')
                               or candidate.get('domain')
                               or prop_info.get('domain_expected', 'Unknown'))
                prop_range  = (ont_info.get('range')
                               or candidate.get('range')
                               or prop_info.get('range_expected', 'Unknown'))

                domain_result      = domain_validator.validate_domain(source_class_code, prop_domain)
                domain_valid       = domain_result['valid']
                domain_match_type  = domain_result['match_type']

                range_checks: List[Dict] = []
                for tgt_rank, tgt_cand in enumerate(target_top_classes, 1):
                    if not isinstance(tgt_cand, dict):
                        continue
                    tgt_code   = tgt_cand.get('code', selected_target_code)
                    range_res  = range_validator.validate_range(tgt_code, prop_range)
                    range_checks.append({
                        'target_rank':            tgt_rank,
                        'target_class_code':      tgt_code,
                        'target_class_label':     tgt_cand.get('label', tgt_code),
                        'target_class_score':     float(tgt_cand.get('score', 0.0)),
                        'range_valid':            range_res['valid'],
                        'range_match_type':       range_res['match_type'],
                        'range_explanation':      range_res['explanation'],
                        'is_selected_target_class': tgt_code == selected_target_code,
                    })

                total_range_checks += len(range_checks)
                valid_rc = [rc for rc in range_checks if rc['range_valid']]
                valid_range_checks += len(valid_rc)

                range_valid = len(valid_rc) > 0
                if not range_valid:
                    range_match_type = 'invalid'
                elif any(rc['range_match_type'] == 'exact' for rc in valid_rc):
                    range_match_type = 'exact'
                else:
                    range_match_type = 'subclass'

                selected_rc        = next((rc for rc in range_checks if rc['is_selected_target_class']), None)
                selected_rng_valid = selected_rc['range_valid'] if selected_rc else False

                is_valid = domain_valid and range_valid
                if is_valid:
                    relation_validity[relation_key] = True

                is_selected = prop_code == selected_property_code
                if is_selected:
                    selected_candidate_validity.append(is_valid)

                validation_data.append({
                    'filename':               filename,
                    'relation_key':           relation_key,
                    'candidate_rank':         cand_rank,
                    'candidate_count':        len(top_candidates),
                    'is_selected_candidate':  is_selected,
                    'source_class':           source_class_code,
                    'target_column':          target_column,
                    'target_class':           selected_target_code,
                    'target_top_classes':     [
                        {'rank': rc['target_rank'], 'code': rc['target_class_code'],
                         'label': rc['target_class_label'], 'score': rc['target_class_score']}
                        for rc in range_checks
                    ],
                    'property_code':          prop_code,
                    'property_label':         prop_label,
                    'confidence':             prop_conf,
                    'domain':                 prop_domain,
                    'range':                  prop_range,
                    'domain_valid':           domain_valid,
                    'domain_match_type':      domain_match_type,
                    'range_valid':            range_valid,
                    'range_match_type':       range_match_type,
                    'range_valid_topk_count': len(valid_rc),
                    'range_total_topk_count': len(range_checks),
                    'selected_range_valid':   selected_rng_valid,
                    'range_checks':           range_checks,
                    'is_source_subclass':     hierarchy_util.is_same_or_subclass(source_class_code, prop_domain),
                    'is_target_subclass':     any(
                        hierarchy_util.is_same_or_subclass(rc['target_class_code'], prop_range)
                        for rc in range_checks
                    ),
                    'valid': is_valid,
                })

        if not validation_data:
            continue

        total_props        = len(validation_data)
        valid_props        = sum(1 for v in validation_data if v['valid'])
        domain_valid_count = sum(1 for v in validation_data if v['domain_valid'])
        range_valid_count  = sum(1 for v in validation_data if v['range_valid'])
        total_relations    = len(relation_validity)
        valid_relations    = sum(1 for ok in relation_validity.values() if ok)
        sel_total          = len(selected_candidate_validity)
        sel_valid          = sum(1 for ok in selected_candidate_validity if ok)

        validation_results[filename] = {
            'validation_data': validation_data,
            'stats': {
                'total_properties':         total_props,
                'valid_properties':         valid_props,
                'invalid_properties':       total_props - valid_props,
                'domain_valid_count':       domain_valid_count,
                'range_valid_count':        range_valid_count,
                'validation_rate':          valid_props   / total_props    if total_props    else 0,
                'domain_validation_rate':   domain_valid_count / total_props if total_props  else 0,
                'range_validation_rate':    range_valid_count  / total_props if total_props  else 0,
                'total_relations':          total_relations,
                'valid_relations':          valid_relations,
                'relation_validation_rate': valid_relations / total_relations if total_relations else 0,
                'selected_candidates_total':  sel_total,
                'selected_candidates_valid':  sel_valid,
                'selected_validation_rate':   sel_valid / sel_total if sel_total else 0,
                'total_range_checks':         total_range_checks,
                'valid_range_checks':         valid_range_checks,
                'range_check_rate':           valid_range_checks / total_range_checks if total_range_checks else 0,
            },
        }

        s = validation_results[filename]['stats']
        print(f"\n[{filename}]  relations: {total_relations}  |  "
              f"valid relations: {valid_relations} ({s['relation_validation_rate']:.1%})  |  "
              f"top-1 property valid: {sel_valid}/{sel_total} ({s['selected_validation_rate']:.1%})")

    return validation_results


def generate_validation_report(validation_results: Dict) -> None:
    """
    Print a structured validation summary to stdout.

    The report includes a per-file summary table, match-type distributions
    for domain and range constraints, and ranked lists of the ten most
    frequently valid and invalid properties across all files.

    Parameters
    ----------
    validation_results : Dict
        Output of :func:`validate_multi_file_mappings`.
    """
    if not validation_results:
        print("No validation results available.")
        return

    print("\n" + "=" * 80)
    print("VALIDATION REPORT")
    print("=" * 80)
    print(f"{'File':<20} {'Total':>7} {'Valid':>7} {'Invalid':>9} {'Rate':>7}")
    print("-" * 80)

    total_all = valid_all = 0
    for filename, res in validation_results.items():
        s = res['stats']
        total_all += s['total_properties']
        valid_all += s['valid_properties']
        print(f"{filename:<20} {s['total_properties']:>7} {s['valid_properties']:>7} "
              f"{s['invalid_properties']:>9} {s['validation_rate']:>7.1%}")

    global_rate = valid_all / total_all if total_all else 0
    print("-" * 80)
    print(f"{'TOTAL':<20} {total_all:>7} {valid_all:>7} {total_all - valid_all:>9} {global_rate:>7.1%}")
    print("=" * 80)

    domain_types: Dict[str, int] = {}
    range_types:  Dict[str, int] = {}
    for res in validation_results.values():
        for v in res['validation_data']:
            domain_types[v['domain_match_type']] = domain_types.get(v['domain_match_type'], 0) + 1
            range_types[v['range_match_type']]   = range_types.get(v['range_match_type'],  0) + 1

    print("\nDomain match types:")
    for t, c in sorted(domain_types.items(), key=lambda x: -x[1]):
        print(f"  {t:<20}: {c:>4} ({c / total_all * 100:.1f}%)")
    print("\nRange match types:")
    for t, c in sorted(range_types.items(), key=lambda x: -x[1]):
        print(f"  {t:<20}: {c:>4} ({c / total_all * 100:.1f}%)")

    def _rank_properties(valid: bool, top_n: int = 10) -> None:
        counter: Dict[str, Dict] = {}
        for res in validation_results.values():
            for v in res['validation_data']:
                if v['valid'] == valid:
                    p = v['property_code']
                    counter.setdefault(p, {'label': v['property_label'], 'count': 0})
                    counter[p]['count'] += 1
        label = "Valid" if valid else "Invalid"
        print(f"\nTop {top_n} {label.lower()} properties:")
        for i, (code, info) in enumerate(
            sorted(counter.items(), key=lambda x: -x[1]['count'])[:top_n], 1
        ):
            print(f"  {i:>2}. {code:<30}  ×{info['count']:>3}")

    _rank_properties(valid=True)
    _rank_properties(valid=False)
    print("=" * 80)


In [21]:
def _replay_validation_for_k(
    mapping_results:  Dict,
    domain_validator: DomainValidator,
    range_validator:  RangeValidator,
    properties_cidoc: Dict,
    k_classes:        int,
    k_properties:     int,
) -> float:
    """
    Replay domain/range validation on pre-computed mapping results using
    truncated top-*k* candidate lists.

    Embedding inference is **not** repeated; only the stored candidate lists
    are sliced to length *k_classes* and *k_properties* before validation.
    A column is considered *covered* if at least one (property candidate,
    target-class candidate) pair satisfies both domain and range constraints.

    Parameters
    ----------
    mapping_results  : Dict  Output of :func:`process_all_csv_files` at MAX_K.
    domain_validator : DomainValidator
    range_validator  : RangeValidator
    properties_cidoc : Dict  Property registry.
    k_classes        : int   Number of class candidates to consider.
    k_properties     : int   Number of property candidates to consider.

    Returns
    -------
    float
        Column-level coverage rate ∈ [0, 1].
    """
    total_columns = covered_columns = 0

    for file_result in mapping_results.values():
        phase1 = file_result.get('phase1_results', {})
        phase2 = file_result.get('phase2_results', {})

        for relation_key, prop_info in phase2.items():
            if '_to_' not in relation_key or not isinstance(prop_info, dict):
                continue

            source_class = prop_info.get('source_class', relation_key.split('_to_', 1)[0])
            target_col   = relation_key.split('_to_', 1)[1]

            tgt_info = phase1.get(target_col, {})
            tgt_candidates = tgt_info.get('top_candidates', [{
                'code':  tgt_info.get('selected_class', 'Unknown'),
                'label': tgt_info.get('selected_label', ''),
                'score': tgt_info.get('confidence', 0.0),
            }])[:k_classes]

            prop_candidates = prop_info.get('top_candidates', [{
                'code':   prop_info.get('selected_property', 'Unknown'),
                'label':  prop_info.get('selected_label', ''),
                'score':  prop_info.get('confidence', 0.0),
                'domain': prop_info.get('domain_expected', 'Unknown'),
                'range':  prop_info.get('range_expected', 'Unknown'),
            }])[:k_properties]

            total_columns += 1
            column_covered = False

            for cand in prop_candidates:
                if not isinstance(cand, dict):
                    continue
                prop_code    = cand.get('code', 'Unknown')
                ont          = properties_cidoc.get(prop_code, {})
                prop_domain  = ont.get('domain')  or cand.get('domain')  or prop_info.get('domain_expected', '')
                prop_range   = ont.get('range')   or cand.get('range')   or prop_info.get('range_expected', '')

                dr = domain_validator.validate_domain(source_class, prop_domain)
                if not dr['valid']:
                    continue
                for tc in tgt_candidates:
                    if not isinstance(tc, dict):
                        continue
                    rr = range_validator.validate_range(tc.get('code', ''), prop_range)
                    if rr['valid']:
                        column_covered = True
                        break
                if column_covered:
                    break

            if column_covered:
                covered_columns += 1

    return covered_columns / total_columns if total_columns else 0.0


def auto_select_top_k(
    dataframes_dict:      Dict,
    cidoc_classes:        List[Dict],
    cidoc_properties:     List[Dict],
    embedder:             MultilingualEmbedder,
    domain_validator:     DomainValidator,
    range_validator:      RangeValidator,
    hierarchy_util:       ClassHierarchyUtil,
    ontology_loader:      CIDOCOntologyLoader,
    use_descriptions:     bool = True,
    column_descriptions:  Optional[Dict] = None,
    max_k_classes:        int = MAX_TOP_K_CLASSES,
    max_k_properties:     int = MAX_TOP_K_PROPERTIES,
    parsimony_threshold:  float = PARSIMONY_THRESHOLD,
) -> tuple:
    """
    Automatically determine the optimal (top_k_classes, top_k_properties) via
    exhaustive grid search over [1, max_k_classes] × [1, max_k_properties].

    Algorithm
    ---------
    1. Run Phase 1 and Phase 2 once at (max_k_classes, max_k_properties) to
       obtain the full ranked candidate lists.  Sentence-embedding inference
       is performed **exactly once**.
    2. For each (k_c, k_p) pair, replay domain/range validation by truncating
       the pre-computed candidate lists.  Record the column-level coverage
       rate (fraction of columns with ≥1 valid RDF triple).
    3. Select the lexicographically smallest pair (k_c*, k_p*) whose coverage
       satisfies: coverage ≥ max_coverage × (1 − parsimony_threshold).

    Parameters
    ----------
    dataframes_dict      : Dict
    cidoc_classes        : List[Dict]
    cidoc_properties     : List[Dict]
    embedder             : MultilingualEmbedder
    domain_validator     : DomainValidator
    range_validator      : RangeValidator
    hierarchy_util       : ClassHierarchyUtil
    ontology_loader      : CIDOCOntologyLoader
    use_descriptions     : bool
    column_descriptions  : Optional[Dict]
    max_k_classes        : int  Upper bound for k_classes search.
    max_k_properties     : int  Upper bound for k_properties search.
    parsimony_threshold  : float  ε for near-optimal acceptance (0 = strict max).

    Returns
    -------
    optimal_k_classes    : int
    optimal_k_properties : int
    grid_df              : pd.DataFrame  Full (k_c, k_p, coverage_rate) grid.
    precomputed_mapping  : Dict  Mapping results at MAX_K (for reference).
    """
    total_pairs = max_k_classes * max_k_properties
    print(f"Auto top-k selection — search space: "
          f"k_c ∈ [1,{max_k_classes}] × k_p ∈ [1,{max_k_properties}] = {total_pairs} pairs  (ε={parsimony_threshold})")

    # Step 1 — single embedding pass at MAX_K
    print(f"[1/3] Computing embeddings at MAX_K ({max_k_classes}, {max_k_properties}) …")
    precomputed_mapping = process_all_csv_files(
        dataframes_dict=dataframes_dict,
        cidoc_classes=cidoc_classes,
        cidoc_properties=cidoc_properties,
        embedder=embedder,
        top_k_classes=max_k_classes,
        top_k_properties=max_k_properties,
        use_descriptions=use_descriptions,
        column_descriptions=column_descriptions,
    )
    p_cidoc = globals().get('properties_cidoc', {})

    # Step 2 — grid search (validation replay only)
    print(f"[2/3] Grid search over {total_pairs} pairs …")
    records   = []
    evaluated = 0
    for k_c in range(1, max_k_classes + 1):
        for k_p in range(1, max_k_properties + 1):
            cov = _replay_validation_for_k(
                mapping_results=precomputed_mapping,
                domain_validator=domain_validator,
                range_validator=range_validator,
                properties_cidoc=p_cidoc,
                k_classes=k_c,
                k_properties=k_p,
            )
            records.append({'k_classes': k_c, 'k_properties': k_p, 'coverage_rate': cov})
            evaluated += 1
            if evaluated % 50 == 0 or evaluated == total_pairs:
                print(f"   {evaluated}/{total_pairs} evaluated", end="\r")
    print()

    grid_df      = pd.DataFrame(records)
    max_coverage = grid_df['coverage_rate'].max()
    threshold    = max_coverage * (1.0 - parsimony_threshold) if max_coverage else 0.0

    # Step 3 — parsimonious selection
    best = (
        grid_df[grid_df['coverage_rate'] >= threshold]
        .sort_values(['k_classes', 'k_properties'])
        .iloc[0]
    )
    k_c_opt, k_p_opt = int(best['k_classes']), int(best['k_properties'])

    print(f"[3/3] Optimal pair: k_classes={k_c_opt}, k_properties={k_p_opt}  "
          f"(coverage={float(best['coverage_rate']):.4f}, max={max_coverage:.4f})")
    return k_c_opt, k_p_opt, grid_df, precomputed_mapping


In [22]:
def process_all_csv_files(
    dataframes_dict:     Dict,
    cidoc_classes:       List[Dict],
    cidoc_properties:    List[Dict],
    embedder:            MultilingualEmbedder,
    top_k_classes:       int = 5,
    top_k_properties:    int = 25,
    use_descriptions:    bool = True,
    column_descriptions: Optional[Dict] = None,
) -> Dict:
    """
    Execute Phase 1 and Phase 2 for all CSV files in *dataframes_dict*.

    Parameters
    ----------
    dataframes_dict     : Dict[str, pd.DataFrame]
        Input datasets indexed by filename.
    cidoc_classes       : List[Dict]
    cidoc_properties    : List[Dict]
    embedder            : MultilingualEmbedder
    top_k_classes       : int  Number of CIDOC class candidates per column.
    top_k_properties    : int  Number of CIDOC property candidates per relation.
    use_descriptions    : bool
    column_descriptions : Optional[Dict]

    Returns
    -------
    Dict
        Per-file results containing 'phase1_results', 'phase2_results',
        'dataframe', and 'stats'.  Files that raise an exception are stored
        with an 'error' key instead.
    """
    all_results: Dict = {}

    for idx, (filename, df) in enumerate(dataframes_dict.items(), 1):
        print(f"\n[{idx}/{len(dataframes_dict)}] Processing {filename} "
              f"({df.shape[0]} rows × {df.shape[1]} columns)")
        try:
            phase1_results, avg_conf_classes = phase1_class_selection(
                df=df,
                cidoc_classes=cidoc_classes,
                embedder=embedder,
                top_k=top_k_classes,
                use_descriptions=use_descriptions,
                column_descriptions=column_descriptions,
            )
            phase2_results = phase2_property_selection(
                phase1_results=phase1_results,
                cidoc_properties=cidoc_properties,
                embedder=embedder,
                top_k=top_k_properties,
                use_descriptions=use_descriptions,
                column_descriptions=column_descriptions,
            )
            avg_conf_props = float(np.mean(
                [r['confidence'] for r in phase2_results.values()]
            )) if phase2_results else 0.0

            all_results[filename] = {
                'dataframe':       df,
                'phase1_results':  phase1_results,
                'phase2_results':  phase2_results,
                'stats': {
                    'num_rows':                  df.shape[0],
                    'num_columns':               df.shape[1],
                    'avg_confidence_classes':    avg_conf_classes,
                    'avg_confidence_properties': avg_conf_props,
                    'classes_mapped':            len(phase1_results),
                    'properties_mapped':         len(phase2_results),
                },
            }
        except Exception as exc:
            print(f"  [ERROR] {filename}: {exc}")
            all_results[filename] = {'error': str(exc), 'stats': None}

    return all_results


In [23]:
def run_complete_multi_file_pipeline(
    dataframes_dict:     Dict,
    cidoc_classes:       List[Dict],
    cidoc_properties:    List[Dict],
    embedder:            MultilingualEmbedder,
    domain_validator:    DomainValidator,
    range_validator:     RangeValidator,
    hierarchy_util:      ClassHierarchyUtil,
    ontology_loader:     CIDOCOntologyLoader,
    top_k_classes:       int = 5,
    top_k_properties:    int = 25,
    use_descriptions:    bool = True,
    column_descriptions: Optional[Dict] = None,
    export_results:      bool = True,
    visualize:           bool = True,
) -> Dict:
    """
    Full ontology-population pipeline: mapping → validation → summary.

    Steps
    -----
    1. **Mapping** : run Phase 1 (class selection) and Phase 2 (property
       selection) for all input files via :func:`process_all_csv_files`.
    2. **Validation** : verify domain and range constraints for all candidate
       triples via :func:`validate_multi_file_mappings`.
    3. **Summary** : print aggregate statistics and quality indicators.

    Parameters
    ----------
    dataframes_dict     : Dict[str, pd.DataFrame]
    cidoc_classes       : List[Dict]
    cidoc_properties    : List[Dict]
    embedder            : MultilingualEmbedder
    domain_validator    : DomainValidator
    range_validator     : RangeValidator
    hierarchy_util      : ClassHierarchyUtil
    ontology_loader     : CIDOCOntologyLoader
    top_k_classes       : int
    top_k_properties    : int
    use_descriptions    : bool
    column_descriptions : Optional[Dict]
    export_results      : bool  Reserved for future export integration.
    visualize           : bool  Reserved for future visualisation integration.

    Returns
    -------
    Dict
        Keys: 'mapping' (:func:`process_all_csv_files` output),
        'validation' (:func:`validate_multi_file_mappings` output).
    """
    mapping_results = process_all_csv_files(
        dataframes_dict=dataframes_dict,
        cidoc_classes=cidoc_classes,
        cidoc_properties=cidoc_properties,
        embedder=embedder,
        top_k_classes=top_k_classes,
        top_k_properties=top_k_properties,
        use_descriptions=use_descriptions,
        column_descriptions=column_descriptions,
    )

    validation_results = validate_multi_file_mappings(
        all_results=mapping_results,
        domain_validator=domain_validator,
        range_validator=range_validator,
        hierarchy_util=hierarchy_util,
        ontology_loader=ontology_loader,
        properties_cidoc=globals().get('properties_cidoc', {}),
    )

    # Aggregate summary
    successful = sum(1 for r in mapping_results.values() if 'error' not in r)
    total_valid   = sum(r['stats']['valid_properties']   for r in validation_results.values())
    total_invalid = sum(r['stats']['invalid_properties'] for r in validation_results.values())
    global_rate   = total_valid / (total_valid + total_invalid) * 100 if (total_valid + total_invalid) else 0
    avg_conf      = float(np.mean([
        r['stats']['avg_confidence_classes']
        for r in mapping_results.values() if r.get('stats')
    ]))

    print(f"\nPipeline summary — {successful}/{len(mapping_results)} files  |  "
          f"valid properties: {total_valid} ({global_rate:.1f}%)  |  "
          f"mean class confidence: {avg_conf:.4f}")

    return {'mapping': mapping_results, 'validation': validation_results}


# ── Pipeline execution ────────────────────────────────────────────────────────

required = {
    'dataframes':      lambda: 'dataframes'     in globals() and len(dataframes) > 0,
    'embedder':        lambda: 'embedder'        in globals() and embedder is not None,
    'cidoc_classes':   lambda: 'cidoc_classes'   in globals() and len(cidoc_classes) > 0,
    'cidoc_properties':lambda: 'cidoc_properties'in globals() and len(cidoc_properties) > 0,
    'domain_validator':lambda: 'domain_validator'in globals(),
    'range_validator': lambda: 'range_validator' in globals(),
    'hierarchy_util':  lambda: 'hierarchy_util'  in globals(),
    'ontology_loader': lambda: 'ontology_loader' in globals(),
}
missing = [k for k, check in required.items() if not check()]

if missing:
    print(f"Cannot run pipeline, missing components: {missing}")
else:
    # Step 0: automatic top-k selection
    TOP_K_CLASSES, TOP_K_PROPERTIES, topk_grid_df, precomputed_mapping = auto_select_top_k(
        dataframes_dict=dataframes,
        cidoc_classes=cidoc_classes,
        cidoc_properties=cidoc_properties,
        embedder=embedder,
        domain_validator=domain_validator,
        range_validator=range_validator,
        hierarchy_util=hierarchy_util,
        ontology_loader=ontology_loader,
        use_descriptions=USE_COLUMN_DESCRIPTIONS,
        column_descriptions=COLUMN_DESCRIPTIONS,
        max_k_classes=MAX_TOP_K_CLASSES,
        max_k_properties=MAX_TOP_K_PROPERTIES,
        parsimony_threshold=PARSIMONY_THRESHOLD,
    )

    # Step 1–3: full pipeline with auto-selected k values
    pipeline_results = run_complete_multi_file_pipeline(
        dataframes_dict=dataframes,
        cidoc_classes=cidoc_classes,
        cidoc_properties=cidoc_properties,
        embedder=embedder,
        domain_validator=domain_validator,
        range_validator=range_validator,
        hierarchy_util=hierarchy_util,
        ontology_loader=ontology_loader,
        top_k_classes=TOP_K_CLASSES,
        top_k_properties=TOP_K_PROPERTIES,
        use_descriptions=USE_COLUMN_DESCRIPTIONS,
        column_descriptions=COLUMN_DESCRIPTIONS,
    )


Auto top-k selection — search space: k_c ∈ [1,15] × k_p ∈ [1,30] = 450 pairs  (ε=0.0)
[1/3] Computing embeddings at MAX_K (15, 30) …

[1/1] Processing us.csv (181 rows × 11 columns)


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Phase 1 complete — 11 columns mapped, mean confidence: 0.3903


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Phase 2 complete — 11 relations mapped, mean confidence: 0.5758
[2/3] Grid search over 450 pairs …
   450/450 evaluated
[3/3] Optimal pair: k_classes=2, k_properties=27  (coverage=1.0000, max=1.0000)

[1/1] Processing us.csv (181 rows × 11 columns)


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Phase 1 complete — 11 columns mapped, mean confidence: 0.3903


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Phase 2 complete — 11 relations mapped, mean confidence: 0.5758

[us.csv]  relations: 11  |  valid relations: 11 (100.0%)  |  top-1 property valid: 3/11 (27.3%)

Pipeline summary — 1/1 files  |  valid properties: 31 (10.4%)  |  mean class confidence: 0.3903


In [24]:
import os, json
from datetime import datetime

if 'pipeline_results' not in globals():
    raise RuntimeError("'pipeline_results' not found. Run the pipeline cell first.")

mapping_results         = pipeline_results.get('mapping', {})
validation_results_exp  = pipeline_results.get('validation', {})

if not mapping_results:
    raise RuntimeError("No mapping results found in pipeline_results['mapping'].")

timestamp       = datetime.now().strftime("%Y%m%d_%H%M%S")
base_output_dir = os.path.join("../results", f"export_{timestamp}")
os.makedirs(base_output_dir, exist_ok=True)

# Flatten Phase 1 results
phase1_selected_rows, phase1_topk_rows, all_dataset_columns = [], [], []
for filename, file_result in mapping_results.items():
    for col, item in file_result.get('phase1_results', {}).items():
        sel = item.get('selected_class', '')
        all_dataset_columns.append({'filename': filename, 'column_name': col})
        phase1_selected_rows.append({
            'filename': filename, 'column_name': col,
            'selected_class': sel, 'selected_label': item.get('selected_label', ''),
            'selected_confidence': item.get('confidence'),
            'justification': item.get('justification', ''),
            'embedding_text': item.get('embedding_text', ''),
        })
        for rank, cand in enumerate(item.get('top_candidates', []), 1):
            phase1_topk_rows.append({
                'filename': filename, 'column_name': col, 'rank': rank,
                'class_code': cand.get('code', ''), 'class_label': cand.get('label', ''),
                'score': cand.get('score'), 'is_selected': cand.get('code', '') == sel,
            })

# Flatten Phase 2 results
phase2_selected_rows, phase2_topk_rows = [], []
for filename, file_result in mapping_results.items():
    for rel_key, item in file_result.get('phase2_results', {}).items():
        sel = item.get('selected_property', '')
        phase2_selected_rows.append({
            'filename': filename, 'relation_key': rel_key,
            'source_class': item.get('source_class', ''),
            'target_column': item.get('target_column', ''),
            'selected_property': sel, 'selected_label': item.get('selected_label', ''),
            'selected_confidence': item.get('confidence'),
            'domain_expected': item.get('domain_expected', ''),
            'range_expected': item.get('range_expected', ''),
            'relation_text': item.get('relation_text', ''),
        })
        for rank, cand in enumerate(item.get('top_candidates', []), 1):
            phase2_topk_rows.append({
                'filename': filename, 'relation_key': rel_key,
                'source_class': item.get('source_class', ''),
                'target_column': item.get('target_column', ''),
                'rank': rank, 'property_code': cand.get('code', ''),
                'property_label': cand.get('label', ''), 'score': cand.get('score'),
                'domain': cand.get('domain', ''), 'range': cand.get('range', ''),
                'is_selected': cand.get('code', '') == sel,
            })

# Flatten validation results
validation_candidates_rows, validation_range_rows, triplets_rows = [], [], []
for filename, file_result in validation_results_exp.items():
    for item in file_result.get('validation_data', []):
        rel_key = item.get('relation_key', '')
        src     = item.get('source_class', '')
        tgt_col = item.get('target_column', '')
        p_code  = item.get('property_code', '')
        p_label = item.get('property_label', '')

        validation_candidates_rows.append({
            'filename': filename, 'relation_key': rel_key,
            'candidate_rank': item.get('candidate_rank'),
            'candidate_count': item.get('candidate_count'),
            'is_selected_candidate': item.get('is_selected_candidate', False),
            'source_class': src, 'target_column': tgt_col,
            'target_class_selected': item.get('target_class', ''),
            'property_code': p_code, 'property_label': p_label,
            'confidence': item.get('confidence'),
            'domain': item.get('domain', ''), 'range': item.get('range', ''),
            'domain_valid': item.get('domain_valid', False),
            'domain_match_type': item.get('domain_match_type', ''),
            'range_valid': item.get('range_valid', False),
            'range_match_type': item.get('range_match_type', ''),
            'valid': item.get('valid', False),
        })

        for rc in item.get('range_checks', []):
            row = {
                'filename': filename, 'relation_key': rel_key,
                'source_class': src, 'target_column': tgt_col,
                'property_code': p_code, 'property_label': p_label,
                'candidate_rank': item.get('candidate_rank'),
                'target_class_code': rc.get('target_class_code', ''),
                'target_class_label': rc.get('target_class_label', ''),
                'target_rank': rc.get('target_rank'),
                'target_class_score': rc.get('target_class_score'),
                'domain_valid': item.get('domain_valid', False),
                'range_valid': rc.get('range_valid', False),
                'pair_valid': bool(item.get('domain_valid')) and bool(rc.get('range_valid')),
                'domain_match_type': item.get('domain_match_type', ''),
                'range_match_type': rc.get('range_match_type', ''),
                'is_selected_target_class': rc.get('is_selected_target_class', False),
            }
            validation_range_rows.append(row)
            if row['pair_valid']:
                triplets_rows.append({
                    'filename': filename, 'relation_key': rel_key,
                    'source_class': src, 'target_column': tgt_col,
                    'property_code': p_code, 'property_label': p_label,
                    'target_class_code': row['target_class_code'],
                    'target_class_label': row['target_class_label'],
                    'candidate_rank': row['candidate_rank'],
                    'target_rank': row['target_rank'],
                    'domain_match_type': row['domain_match_type'],
                    'range_match_type': row['range_match_type'],
                })

# Column-level coverage
all_cols_df     = pd.DataFrame(all_dataset_columns).drop_duplicates()
triplets_df     = pd.DataFrame(triplets_rows)

if triplets_df.empty:
    covered_counts = pd.DataFrame(columns=['filename', 'target_column', 'nb_valid_triplets'])
else:
    covered_counts = (
        triplets_df.groupby(['filename', 'target_column'], as_index=False)
        .size().rename(columns={'size': 'nb_valid_triplets'})
    )

coverage_df = all_cols_df.merge(
    covered_counts, how='left',
    left_on=['filename', 'column_name'],
    right_on=['filename', 'target_column'],
).drop(columns=['target_column'])
coverage_df['nb_valid_triplets'] = coverage_df['nb_valid_triplets'].fillna(0).astype(int)
coverage_df['is_covered']        = coverage_df['nb_valid_triplets'] > 0

# Per-dataset statistics
coverage_stats = []
for fname, grp in coverage_df.groupby('filename'):
    covered     = sorted(grp.loc[grp['is_covered'],  'column_name'].dropna().tolist())
    not_covered = sorted(grp.loc[~grp['is_covered'], 'column_name'].dropna().tolist())
    coverage_stats.append({
        'filename':                fname,
        'total_columns':           len(grp),
        'covered_columns_count':   len(covered),
        'not_covered_columns_count': len(not_covered),
        'coverage_rate':           len(covered) / len(grp) if grp.shape[0] else 0,
        'covered_columns':         ' | '.join(covered),
        'not_covered_columns':     ' | '.join(not_covered),
    })
coverage_stats_df = pd.DataFrame(coverage_stats)

# Write CSV exports
exports = {
    'phase1_selected.csv':          pd.DataFrame(phase1_selected_rows),
    'phase1_topk.csv':              pd.DataFrame(phase1_topk_rows),
    'phase2_selected.csv':          pd.DataFrame(phase2_selected_rows),
    'phase2_topk.csv':              pd.DataFrame(phase2_topk_rows),
    'validation_candidates.csv':    pd.DataFrame(validation_candidates_rows),
    'validation_range_checks.csv':  pd.DataFrame(validation_range_rows),
    'valid_triplets.csv':           triplets_df,
    'column_coverage_details.csv':  coverage_df,
    'dataset_coverage_summary.csv': coverage_stats_df,
}
for fname, df_out in exports.items():
    df_out.to_csv(os.path.join(base_output_dir, fname), index=False, encoding='utf-8')

# Write JSON summary
summary = {
    'export_timestamp': timestamp,
    'output_dir':       base_output_dir,
    'selected_top_k': {
        'k_classes':    int(TOP_K_CLASSES)    if 'TOP_K_CLASSES'    in globals() else None,
        'k_properties': int(TOP_K_PROPERTIES) if 'TOP_K_PROPERTIES' in globals() else None,
    },
    'counts': {k: len(v) for k, v in exports.items()},
    'dataset_coverage': coverage_stats_df.to_dict(orient='records'),
}
with open(os.path.join(base_output_dir, 'export_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"Export complete — {base_output_dir}")


Export complete — ../results/export_20260417_200052


In [25]:
import os, json

if 'pipeline_results' in globals() and 'validation' in pipeline_results:
    _validation = pipeline_results['validation']
elif 'validation_results' in globals():
    _validation = validation_results
else:
    raise RuntimeError("No validation results found. Run the pipeline cell first.")

if 'cidoc_classes' not in globals() or 'cidoc_properties' not in globals():
    raise RuntimeError("cidoc_classes / cidoc_properties not found. Load CIDOC data first.")

# Build scope-note indexes for classes and properties
class_notes    = {c['code'].strip(): (c.get('description') or '').strip() for c in cidoc_classes}
property_notes = {p['code'].strip(): (p.get('description') or '').strip() for p in cidoc_properties}

# Collect only ontologically valid (subject, property, object) triples
classes_out:    Dict = {}
properties_out: Dict = {}
relations_out:  List = []
seen:           set  = set()

for file_result in _validation.values():
    for item in file_result.get('validation_data', []):
        src   = (item.get('source_class')   or '').strip()
        p     = (item.get('property_code')  or '').strip()
        d_ok  = bool(item.get('domain_valid', False))
        range_checks = item.get('range_checks') or []

        candidates = range_checks if range_checks else [{
            'target_class_code': (item.get('target_class') or '').strip(),
            'range_valid':       bool(item.get('range_valid', False)),
        }]

        for rc in candidates:
            tgt    = (rc.get('target_class_code') or '').strip()
            r_ok   = bool(rc.get('range_valid', False))
            triple = (src, p, tgt)

            if not (d_ok and r_ok and all(triple) and triple not in seen):
                continue
            seen.add(triple)

            classes_out.setdefault(src, {'description': class_notes.get(src, '')})
            classes_out.setdefault(tgt, {'description': class_notes.get(tgt, '')})
            properties_out.setdefault(p, {'description': property_notes.get(p, '')})
            relations_out.append({'subject': src, 'property': p, 'object': tgt})

ontology_subset = {
    'classes':    dict(sorted(classes_out.items())),
    'properties': dict(sorted(properties_out.items())),
    'relations':  relations_out,
}

output_dir  = '../results'
output_file = os.path.join(output_dir, 'ontology_subset.json')
os.makedirs(output_dir, exist_ok=True)
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(ontology_subset, f, ensure_ascii=False, indent=2)

print(f"Ontology subset written to {output_file}")
print(f"  Classes: {len(ontology_subset['classes'])}  "
      f"Properties: {len(ontology_subset['properties'])}  "
      f"Relations: {len(ontology_subset['relations'])}")


Ontology subset written to ../results/ontology_subset.json
  Classes: 11  Properties: 12  Relations: 22
